# Kreol ASR harness (Colab)

Opens a web app for testing Meta's Omnilingual ASR on Kreol Morisien.

1. **Runtime → Change runtime type → T4 GPU** (free).
2. **Runtime → Run all.**
3. Open the `https://….gradio.live` link printed by the last cell. It works on your phone too, so you can record voice notes directly.

The first run of each model downloads it (300M ≈ 1 GB, 1B ≈ 4 GB). 7B models won't fit on a free T4.
The install downgrades torch and numpy to the versions Meta needs, so Colab will likely ask you to **restart the session**. Do it, then **Run all** again (the second install is instant).

### 1. Keep logs on Google Drive (optional)
Runs and audio go to `MyDrive/kreol-asr-runs`. Skip this cell to keep them only for this session.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['KREOL_LOG_DIR'] = '/content/drive/MyDrive/kreol-asr-runs'
except Exception as e:
    print('Drive not mounted, logging locally:', e)

### 2. Install

In [ ]:
# Colab runs Python 3.12.x; Meta declares <=3.12, which pip reads as 3.12.0 only.
!pip install -q --ignore-requires-python 'omnilingual-asr==0.2.0' 'torch==2.8.0' 'torchaudio==2.8.0' 'pyarrow>=20,<21' 'gradio>=5,<6' 'jiwer>=3' 'soundfile>=0.12'
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 3. Harness code

In [ ]:
%%writefile asr_core.py
"""Core helpers for the Kreol ASR web harness: audio prep, splitting, phone
simulation, scoring and a cached wrapper around Meta's omnilingual-asr models.

Everything except `ModelBank` runs without the model package, so it can be
tested anywhere.
"""

from __future__ import annotations

import re
import time
import unicodedata
from dataclasses import dataclass

import numpy as np

SR = 16_000
MAX_CLIP_S = 40.0  # hard limit in omnilingual-asr for non-Unlimited models
TARGET_CHUNK_S = 30.0  # aim for chunks well under the limit

# UI label -> model card name, from the omnilingual-asr 0.2.0 cards.
MODELS = {
    "CTC 300M": "omniASR_CTC_300M_v2",
    "CTC 1B": "omniASR_CTC_1B_v2",
    "CTC 3B": "omniASR_CTC_3B_v2",
    "CTC 7B": "omniASR_CTC_7B_v2",
    "LLM 300M": "omniASR_LLM_300M_v2",
    "LLM 1B": "omniASR_LLM_1B_v2",
    "LLM 3B": "omniASR_LLM_3B_v2",
    "LLM 7B": "omniASR_LLM_7B_v2",
    "LLM Unlimited 300M": "omniASR_LLM_Unlimited_300M_v2",
    "LLM Unlimited 1B": "omniASR_LLM_Unlimited_1B_v2",
    "LLM Unlimited 3B": "omniASR_LLM_Unlimited_3B_v2",
    "LLM Unlimited 7B": "omniASR_LLM_Unlimited_7B_v2",
}
DEFAULT_MODELS = ["CTC 300M", "LLM 1B"]

LANGS = {
    "Kreol Morisien (mfe_Latn)": "mfe_Latn",
    "French (fra_Latn)": "fra_Latn",
    "English (eng_Latn)": "eng_Latn",
    "Bhojpuri (bho_Deva)": "bho_Deva",
    "No hint": None,
}


def is_unlimited(card: str) -> bool:
    return "Unlimited" in card


def is_ctc(card: str) -> bool:
    return "_CTC_" in card


# ---------------------------------------------------------------- audio prep


def to_mono_16k(sr: int, wav: np.ndarray) -> np.ndarray:
    """Gradio gives (sr, int16|float array [, channels]). Return float32 mono 16 kHz."""
    wav = np.asarray(wav)
    if wav.dtype.kind == "i":
        wav = wav.astype(np.float32) / np.iinfo(wav.dtype).max
    elif wav.dtype.kind == "u":
        wav = (wav.astype(np.float32) - 128) / 128
    else:
        wav = wav.astype(np.float32)
    if wav.ndim == 2:
        # Gradio uses (samples, channels); guard against (channels, samples).
        wav = wav.mean(axis=1) if wav.shape[0] > wav.shape[1] else wav.mean(axis=0)
    if sr != SR:
        wav = resample(wav, sr, SR)
    return wav


def resample(wav: np.ndarray, sr_in: int, sr_out: int) -> np.ndarray:
    import torch
    import torchaudio.functional as AF

    out = AF.resample(torch.from_numpy(np.ascontiguousarray(wav)), sr_in, sr_out)
    return out.numpy().astype(np.float32)


def simulate_phone(wav: np.ndarray) -> np.ndarray:
    """Rough G.711 phone line: 300–3400 Hz band, 8 kHz, 8-bit mu-law, back to 16 kHz."""
    import torch
    import torchaudio.functional as AF

    x = torch.from_numpy(np.ascontiguousarray(wav))
    x = AF.highpass_biquad(x, SR, 300.0)
    x = AF.lowpass_biquad(x, SR, 3400.0)
    x = AF.resample(x, SR, 8000)
    x = x.clamp(-1, 1)
    x = AF.mu_law_decoding(AF.mu_law_encoding(x, 256), 256)
    x = AF.resample(x, 8000, SR)
    return x.numpy().astype(np.float32)


def split_on_pauses(
    wav: np.ndarray, sr: int = SR, target_s: float = TARGET_CHUNK_S, max_s: float = MAX_CLIP_S - 2
) -> list[tuple[int, int]]:
    """Split into chunks of at most `max_s`, cutting at the quietest 50 ms frame
    between `target_s * 0.6` and `max_s`. Returns (start, end) sample indices."""
    n = len(wav)
    if n <= max_s * sr:
        return [(0, n)]
    hop = int(0.05 * sr)
    frames = n // hop
    energy = np.array([np.sqrt(np.mean(wav[i * hop : (i + 1) * hop] ** 2) + 1e-12) for i in range(frames)])
    bounds, start = [], 0
    while n - start > max_s * sr:
        lo = start + int(target_s * 0.6 * sr)
        hi = start + int(max_s * sr)
        f_lo, f_hi = lo // hop, min(hi // hop, frames)
        cut = (f_lo + int(np.argmin(energy[f_lo:f_hi]))) * hop if f_hi > f_lo else hi
        bounds.append((start, cut))
        start = cut
    bounds.append((start, n))
    return bounds


# ---------------------------------------------------------------- scoring


def normalize(text: str) -> str:
    """Light normalisation for scoring: lowercase, unify apostrophes, drop
    punctuation, collapse spaces. Kreol spelling variants are NOT merged here;
    use the eval repo's variants.tsv for that."""
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"[’‘`´]", "'", text)
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"(?<!\w)'|'(?!\w)", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def score(reference: str, hypothesis: str) -> dict:
    import jiwer

    ref, hyp = normalize(reference), normalize(hypothesis)
    if not ref:
        return {}
    return {"wer": jiwer.wer(ref, hyp or "<empty>"), "cer": jiwer.cer(ref, hyp or " ")}


# ---------------------------------------------------------------- models


@dataclass
class Result:
    label: str
    text: str
    seconds: float
    load_seconds: float
    chunks: int
    error: str | None = None


class ModelBank:
    """Loads omnilingual-asr pipelines lazily and keeps them on the GPU."""

    def __init__(self, mock: bool = False, device: str = "auto", dtype: str = "auto"):
        self.mock = mock
        self.device = pick_device(device)
        self.dtype = dtype
        self.loaded: dict[str, object] = {}

    def _get(self, card: str):
        if card in self.loaded:
            return self.loaded[card], 0.0
        t0 = time.perf_counter()
        if self.mock:
            pipe = MockPipeline(card)
        else:
            pipe = self._load(card, self.device)
        self.loaded[card] = pipe
        return pipe, time.perf_counter() - t0

    def _load(self, card: str, device: str):
        import torch
        from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

        dtypes = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}
        # bf16 on NVIDIA; fp32 elsewhere, which is the safe choice on CPU and MPS.
        dtype = dtypes.get(self.dtype) or (torch.bfloat16 if device == "cuda" else torch.float32)
        return ASRInferencePipeline(model_card=card, device=device, dtype=dtype)

    def unload_all(self) -> None:
        self.loaded.clear()
        try:
            import gc

            import torch

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except ImportError:
            pass

    def transcribe(self, label: str, wav: np.ndarray, lang: str | None) -> Result:
        card = MODELS[label]
        try:
            pipe, load_s = self._get(card)
        except Exception as e:  # noqa: BLE001 - surface any load error in the UI
            return Result(label, "", 0.0, 0.0, 0, f"Load failed: {type(e).__name__}: {e}")
        spans = [(0, len(wav))] if is_unlimited(card) else split_on_pauses(wav)
        chunks = [{"waveform": wav[a:b], "sample_rate": SR} for a, b in spans]
        langs = None if is_ctc(card) or lang is None else [lang] * len(chunks)
        t0 = time.perf_counter()
        try:
            try:
                texts = pipe.transcribe(chunks, lang=langs, batch_size=1)
            except Exception:
                if self.mock or str(getattr(pipe, "device", "")) != "mps":
                    raise
                # Some fairseq2 ops may be missing on Apple's GPU: retry this model on CPU.
                pipe = self.loaded[card] = self._load(card, "cpu")
                t0 = time.perf_counter()
                texts = pipe.transcribe(chunks, lang=langs, batch_size=1)
        except Exception as e:  # noqa: BLE001
            msg = f"{type(e).__name__}: {e}"
            if "out of memory" in msg.lower():
                msg += " — try a smaller model or click 'Unload models'."
            return Result(label, "", time.perf_counter() - t0, load_s, len(chunks), msg)
        return Result(label, " ".join(t.strip() for t in texts), time.perf_counter() - t0, load_s, len(chunks))


def pick_device(device: str = "auto") -> str:
    if device != "auto":
        return device
    try:
        import torch
    except ImportError:
        return "cpu"
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


class MockPipeline:
    """Stands in for a real model so the UI can be tried without a GPU."""

    def __init__(self, card: str):
        self.card = card
        self.device = "cpu"

    def transcribe(self, chunks, lang=None, batch_size=1):
        return [f"[{self.card} mock: {len(c['waveform']) / SR:.1f}s, lang={lang[0] if lang else None}]" for c in chunks]


In [ ]:
%%writefile app.py
"""Kreol ASR web harness: record or upload audio, run any omnilingual-asr model
side by side, score against a reference, and log every run.

    python app.py            # real models (GPU strongly recommended)
    python app.py --share    # also print a public gradio.live link
    python app.py --mock     # fake models, to try the UI without a GPU
"""

from __future__ import annotations

import argparse
import csv
import datetime as dt
import os
from pathlib import Path

import gradio as gr
import numpy as np
import soundfile as sf

from asr_core import DEFAULT_MODELS, LANGS, MODELS, SR, ModelBank, score, simulate_phone, to_mono_16k

COLUMNS = ["model", "transcript", "WER", "CER", "secs", "RTF", "chunks", "note"]
LOG_FIELDS = ["time", "clip", "source", "duration_s", "phone_sim", "lang", "model", "transcript",
              "reference", "wer", "cer", "secs", "load_secs", "chunks", "error", "notes"]


def device_info(device: str) -> str:
    if device == "cuda":
        import torch

        p = torch.cuda.get_device_properties(0)
        return f"Running on GPU: {p.name}, {p.total_memory / 2**30:.0f} GB."
    if device == "mps":
        return "Running on Apple GPU (MPS), falling back to CPU per model if an op is unsupported."
    return "Running on CPU: start with the 300M models."


def build(bank: ModelBank, log_dir: Path) -> gr.Blocks:
    log_dir.mkdir(parents=True, exist_ok=True)
    (log_dir / "audio").mkdir(exist_ok=True)
    log_path = log_dir / "runs.csv"

    def log_rows(rows: list[dict]) -> None:
        new = not log_path.exists()
        with log_path.open("a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
            if new:
                w.writeheader()
            w.writerows(rows)

    def run_one(wav, clip_name, source, models, lang, phone, reference, notes):
        dur = len(wav) / SR
        if phone:
            wav = simulate_phone(wav)
        stamp = dt.datetime.now().strftime("%Y%m%d-%H%M%S-%f")[:-3]
        saved = log_dir / "audio" / f"{stamp}_{Path(clip_name).stem}{'_phone' if phone else ''}.wav"
        sf.write(saved, wav, SR)
        table, logs = [], []
        for label in models:
            r = bank.transcribe(label, wav, LANGS[lang])
            s = score(reference, r.text) if reference.strip() and not r.error else {}
            note = r.error or (f"first load {r.load_seconds:.0f}s" if r.load_seconds >= 1 else "")
            table.append([label, r.text, _pct(s.get("wer")), _pct(s.get("cer")), round(r.seconds, 2),
                          round(r.seconds / dur, 3) if dur else None, r.chunks, note])
            logs.append({"time": stamp, "clip": saved.name, "source": source, "duration_s": round(dur, 2),
                         "phone_sim": phone, "lang": LANGS[lang], "model": MODELS[label], "transcript": r.text,
                         "reference": reference, "wer": s.get("wer"), "cer": s.get("cer"),
                         "secs": round(r.seconds, 3), "load_secs": round(r.load_seconds, 1),
                         "chunks": r.chunks, "error": r.error, "notes": notes})
        log_rows(logs)
        return table, dur, saved

    def transcribe(audio, models, lang, phone, reference, notes):
        if audio is None:
            raise gr.Error("Record or upload a clip first.")
        if not models:
            raise gr.Error("Tick at least one model.")
        sr, data = audio
        wav = to_mono_16k(sr, data)
        if len(wav) < SR * 0.3:
            raise gr.Error("Clip is shorter than 0.3 s.")
        table, dur, saved = run_one(wav, "clip", "single", models, lang, phone, reference, notes)
        summary = f"{dur:.1f}s clip, saved as `{saved.name}`. {device_info(bank.device)}"
        if phone:
            return table, summary, (SR, wav_for_playback(saved))
        return table, summary, None

    def transcribe_batch(files, models, lang, phone, refs_text):
        if not files:
            raise gr.Error("Upload one or more audio files.")
        if not models:
            raise gr.Error("Tick at least one model.")
        refs = parse_refs(refs_text)
        rows = []
        for f in files:
            path = Path(f if isinstance(f, str) else f.name)
            data, sr = sf.read(path, dtype="float32", always_2d=False)
            wav = to_mono_16k(sr, data)
            ref = refs.get(path.name, refs.get(path.stem, ""))
            table, _, _ = run_one(wav, path.name, "batch", models, lang, phone, ref, "")
            rows += [[path.name] + r for r in table]
        return rows, str(log_path)

    with gr.Blocks(title="Kreol ASR harness") as demo:
        gr.Markdown(
            "## Kreol ASR harness\n"
            "Test Meta's Omnilingual ASR on Kreol Morisien. Pick models, record or upload audio, "
            "and compare. Paste what was actually said to get WER/CER. Every run is logged.  \n"
            f"_{device_info(bank.device)} The first run of each model downloads and loads it, which can take a few minutes._"
        )
        with gr.Row():
            models = gr.CheckboxGroup(list(MODELS), value=DEFAULT_MODELS, label="Models",
                                      info="CTC: fast, no language hint. LLM: uses the hint below. "
                                           "Unlimited: no 40 s cap. 7B needs a big GPU.")
        with gr.Row():
            lang = gr.Dropdown(list(LANGS), value=list(LANGS)[0], label="Language hint (LLM models only)")
            phone = gr.Checkbox(False, label="Simulate phone line (8 kHz G.711)",
                                info="Degrades the clip like a phone call before transcribing.")

        with gr.Tab("Single clip"):
            audio = gr.Audio(sources=["upload", "microphone"], type="numpy", label="Clip (any length)")
            reference = gr.Textbox(label="Reference transcript (optional)", lines=2,
                                   placeholder="What was actually said, in Kreol. Leave empty to skip scoring.")
            notes = gr.Textbox(label="Notes (optional, saved to log)", placeholder="speaker, setting, code-switching…")
            go = gr.Button("Transcribe", variant="primary")
            summary = gr.Markdown()
            out = gr.Dataframe(headers=COLUMNS, wrap=True, interactive=False)
            phone_audio = gr.Audio(label="What the model heard (phone-simulated)", interactive=False)
            go.click(transcribe, [audio, models, lang, phone, reference, notes], [out, summary, phone_audio])

        with gr.Tab("Batch"):
            files = gr.File(file_count="multiple", file_types=["audio"], label="Audio files")
            refs = gr.Textbox(label="References (optional)", lines=4,
                              placeholder="one per line:  filename<TAB or |>transcript")
            go_b = gr.Button("Transcribe all", variant="primary")
            out_b = gr.Dataframe(headers=["file"] + COLUMNS, wrap=True, interactive=False)
            log_file = gr.File(label="Full run log (CSV)")
            go_b.click(transcribe_batch, [files, models, lang, phone, refs], [out_b, log_file])

        with gr.Tab("Log"):
            gr.Markdown(f"Runs are appended to `{log_path}`; clips are saved in `{log_dir / 'audio'}`.")
            show = gr.Button("Refresh")
            log_table = gr.Dataframe(wrap=True, interactive=False)
            log_dl = gr.File(label="Download log")
            show.click(lambda: (read_log(log_path), str(log_path) if log_path.exists() else None),
                       None, [log_table, log_dl])

        with gr.Row():
            unload = gr.Button("Unload models (free GPU memory)", size="sm")
            status = gr.Markdown()
            unload.click(lambda: (bank.unload_all(), "Unloaded.")[1], None, status)
    return demo


def wav_for_playback(path: Path) -> np.ndarray:
    data, _ = sf.read(path, dtype="float32")
    return data


def parse_refs(text: str) -> dict[str, str]:
    refs = {}
    for line in (text or "").splitlines():
        for sep in ("\t", "|"):
            if sep in line:
                name, ref = line.split(sep, 1)
                refs[name.strip()] = ref.strip()
                break
    return refs


def read_log(path: Path) -> list[list]:
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as f:
        rows = list(csv.reader(f))
    return rows[-200:] if len(rows) <= 1 else [rows[0]] + rows[1:][-200:]


def _pct(x):
    return None if x is None else f"{100 * x:.1f}%"


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--mock", action="store_true", help="fake models, for trying the UI")
    p.add_argument("--share", action="store_true", help="create a public gradio.live link")
    p.add_argument("--log-dir", default=os.environ.get("KREOL_LOG_DIR", "runs"))
    p.add_argument("--port", type=int, default=7860)
    p.add_argument("--device", default="auto", help="auto, cuda, mps or cpu")
    p.add_argument("--dtype", default="auto", help="auto, bf16, fp16 or fp32")
    a = p.parse_args()
    demo = build(ModelBank(mock=a.mock, device=a.device, dtype=a.dtype), Path(a.log_dir))
    demo.queue(default_concurrency_limit=1).launch(share=a.share, inbrowser=not a.share, server_name="127.0.0.1" if not a.share else "0.0.0.0", server_port=a.port,
                                                   allowed_paths=[str(Path(a.log_dir).resolve())])


if __name__ == "__main__":
    main()


### 4. Launch
Click the `gradio.live` link below. The cell keeps running while the app is up; stop it to shut the app down.

In [ ]:
!python app.py --share --log-dir "${KREOL_LOG_DIR:-runs}"